<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/wip-lunarlander-a2c.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install swig
!pip install gymnasium[box2d]
!pip install tsilva-notebook-utils # Personal utils to help with notebook authoring

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 16.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 965.5/965.5 kB 10.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.4/374.4 kB 13.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for box2d-py: filename=box2d_py-2.3.5-cp310-cp310-linux_x86_64.whl size=2376422 sha256=872f0a7bb1ec1e1f7f0b6a06c5827f6deca4229d000cfcd2ba71b59ac3d5db67
  Stored in directory: /root/.cache/pip/wheels/db/8f/6a/eaaadf056fba10a98d986f6dce954e6201ba3126926fc5ad9e
Successfully built box2d-py
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 12.1 MB/s eta 0:00:00


In [13]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
import matplotlib.pyplot as plt
from collections import deque

# Hyperparameters (updated based on your input)
GAMMA = 0.995          # From gamma: 0.995
LEARNING_RATE = 0.00083  # From learning_rate: lin_0.00083 (assuming linear schedule starts here)
ENTROPY_COEFF = 0.00001  # From ent_coef: 0.00001
MAX_EPISODES = 10_000
SOLVED_SCORE = 200
VALUE_COEFF = 0.5
HIDDEN_LAYER = 256
TOTAL_TIMESTEPS_LIMIT = int(2e5)  # From n_timesteps: 2e5 (we'll use this as a stopping condition)

# Unused from your list:
# - n_envs: 8 (requires vectorized environments, not implemented here)
# - n_steps: 5 (specific to PPO rollout buffers, not applicable here)
# - policy: 'MlpPolicy' (implies an MLP, which we already have)

# Device configuration
DEVICE = "cpu"  # Change to "cuda" if available and desired
print(f"Using device: {DEVICE}")

class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(ActorCritic, self).__init__()
        self.fc1 = nn.Linear(state_dim, HIDDEN_LAYER)
        self.fc2 = nn.Linear(HIDDEN_LAYER, HIDDEN_LAYER)
        self.actor = nn.Linear(HIDDEN_LAYER, action_dim)
        self.critic = nn.Linear(HIDDEN_LAYER, 1)

        for layer in self.modules():
            if isinstance(layer, nn.Linear):
                nn.init.orthogonal_(layer.weight, gain=np.sqrt(2))
                layer.bias.data.zero_()

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        action_probs = F.softmax(self.actor(x), dim=-1)
        state_value = self.critic(x)
        return action_probs, state_value

def train(model):
    # Note: Using LunarLander-v3 as in original code (not v2 as in your hyperparams)
    env = gym.make("LunarLander-v3")
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    model = ActorCritic(state_dim, action_dim).to(DEVICE)

    model.train()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    scores = []
    avg_scores = []
    recent_scores = deque(maxlen=100)
    timesteps_history = []
    total_timesteps = 0

    for episode in range(1, MAX_EPISODES + 1):
        state, _ = env.reset()
        score = 0
        done = False
        episode_timesteps = 0

        states = []
        actions = []
        rewards = []
        dones = []

        while not done:
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)

            with torch.no_grad():
                action_probs, state_value = model(state_tensor)

            dist = Categorical(action_probs)
            action = dist.sample().item()

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            states.append(state)
            actions.append(action)
            rewards.append(reward)
            dones.append(done)

            state = next_state
            score += reward
            episode_timesteps += 1
            total_timesteps += 1

            # Check total timestep limit
            #if total_timesteps >= TOTAL_TIMESTEPS_LIMIT:
            #    print(f"\nReached total timestep limit of {TOTAL_TIMESTEPS_LIMIT}")
            #    break

        #if total_timesteps >= TOTAL_TIMESTEPS_LIMIT:
        #    break

        # Training logic
        states_tensor = torch.FloatTensor(np.array(states)).to(DEVICE)
        actions_tensor = torch.LongTensor(actions).to(DEVICE)
        action_probs, state_values = model(states_tensor)
        state_values = state_values.squeeze()

        dist = Categorical(action_probs)
        log_probs = dist.log_prob(actions_tensor)
        entropy = dist.entropy().mean()

        returns = []
        discounted_reward = 0
        for reward, done in zip(reversed(rewards), reversed(dones)):
            if done:
                discounted_reward = 0
            discounted_reward = reward + GAMMA * discounted_reward
            returns.insert(0, discounted_reward)

        returns = torch.FloatTensor(returns).to(DEVICE)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        advantage = returns - state_values.detach()

        actor_loss = -(log_probs * advantage).mean()
        critic_loss = VALUE_COEFF * F.mse_loss(state_values, returns)
        entropy_loss = -ENTROPY_COEFF * entropy
        loss = actor_loss + critic_loss + entropy_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        scores.append(score)
        recent_scores.append(score)
        avg_score = np.mean(recent_scores)
        avg_scores.append(avg_score)
        timesteps_history.append(episode_timesteps)

        if episode % 10 == 0:
            print(f"Episode {episode}, Score: {score:.2f}, Avg Score (100): {avg_score:.2f}, "
                  f"Total Timesteps: {total_timesteps}, "
                  f"Loss: {loss.item():.4f}")

        if len(recent_scores) == 100 and avg_score >= SOLVED_SCORE:
            print(f"\nEnvironment solved in {episode} episodes!")
            print(f"Average Score: {avg_score:.2f}")
            print(f"Total Timesteps: {total_timesteps}")
            torch.save(model.state_dict(), 'lunar_lander_solved.pt')
            break

    # Plotting
    plt.figure(figsize=(12, 8))

    plt.subplot(2, 1, 1)
    plt.plot(scores, alpha=0.3, color='gray')
    plt.plot(avg_scores, color='blue', linewidth=2)
    plt.xlabel('Episode')
    plt.ylabel('Score')
    plt.title('Learning Curve')
    plt.axhline(y=200, color='r', linestyle='-', alpha=0.5)
    plt.text(len(scores)-100, 210, 'Solved Threshold')

    plt.subplot(2, 1, 2)
    plt.plot(timesteps_history, color='green', linewidth=2)
    plt.xlabel('Episode')
    plt.ylabel('Timesteps')
    plt.title('Timesteps per Episode')

    plt.tight_layout()
    plt.savefig('learning_curves.png')
    plt.show()

    print(f"\nTraining completed!")
    print(f"Total Episodes: {episode}")
    print(f"Total Timesteps: {total_timesteps}")

    return model

env = gym.make("LunarLander-v3")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n
model = ActorCritic(state_dim, action_dim).to(DEVICE)
train(model)

Using device: cpu


KeyboardInterrupt: 

In [ ]:
from tsilva_notebook_utils import render_video

def run_episode(model):
    # Create environment with rendering enabled
    env = gym.make("LunarLander-v3", render_mode="rgb_array")

    # Set model to evaluation mode
    model.eval()

    # Reset environment
    state, _ = env.reset()
    score = 0
    done = False

    # Run the episode
    frames = []
    while not done:
        # Render the environment
        frames.append(env.render())

        # Convert state to tensor
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)

        # Get action probabilities (no gradients needed)
        with torch.no_grad():
            action_probs, _ = model(state_tensor)

        # Sample action from the policy
        dist = Categorical(action_probs)
        action = dist.sample().item()

        # Take action in the environment
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # Update state and score
        state = next_state
        score += reward

    # Close the environment
    env.close()

    print(f"Episode completed! Score: {score:.2f}")

    return score, frames

# Run a single episode with rendering
score, frames = run_episode(model)
render_video(frames)